# Chapter 07: Compression & Hierarchy
### Partial Pooling and the Information Bottleneck

> *"The purpose of abstraction is not to be vague, but to create a new
> semantic level, in which one can be absolutely precise."*
> — Edsger W. Dijkstra

In the previous chapter, we explored Markov Chain Monte Carlo (MCMC) as an
ergodic sampling engine navigating the typical set where parameter probability
density balances state-space volume. We analyzed individual test endpoints or
fixed parameter vectors.

In production engineering, however, tests never live in isolation. A modern
monorepo or microservice architecture may run $J = 50,000$ individual automated
tests across 80 subsystems, compiled by 500 developers.
- Some tests have run $N_j = 10,000$ times with massive historical data.
- Other tests are brand-new, created 20 minutes ago on a pull request, having
  run
  exactly $N_j = 3$ times.

If a new test fails 1 out of 3 times, is its true failure probability $\theta_j
= 0.333$? If you treat every test independently (no pooling), you fall victim to
the tyranny of small sample sizes: noise dominates your alerts, and developers
are inundated with false alarms. Conversely, if you average all 50,000 tests
together into a single global fleet average $\bar{\theta}$ (complete pooling),
you are blind to rogue subsystems whose underlying services are genuinely
broken.

How do we resolve this tension?
1. In **Bayesian Statistics**, the answer is **Hierarchical Modeling and Partial
   Pooling (Shrinkage)**: tests borrow statistical strength from the fleet while
   retaining individual autonomy.
2. In **Information Theory**, the answer is **Naftali Tishby's Information
   Bottleneck Principle**: find a compressed representation $T$ that discards
   nuisance entropy $I(X; T)$ while retaining maximum predictive mutual
   information $I(T; Y)$ about test failure.

This chapter unifies these two profound ideas. We will discover that Bayesian
shrinkage is an optimal lossy compression algorithm governed by Rate-Distortion
theory.



## 1. The Dilemma of the Fleet: No Pooling vs Complete Pooling

Consider $J$ distinct test cases in our continuous integration suite. For each
test $j \in \{1, \dots, J\}$, we observe $k_j$ failures across $n_j$ runs:

$$
k_j \sim \text{Binomial}(n_j, \theta_j)
$$


We face two classical extremes:

### Extreme A: Complete Pooling (The Underfitting Extremum)
Assume all tests share an identical failure probability:

$$
\theta_1 = \theta_2 = \dots = \theta_J = \theta_{\text{fleet}}
$$

$$
\hat{\theta}_{\text{pool}} = \frac{\sum_{j=1}^J k_j}{\sum_{j=1}^J n_j}
$$


- **Information cost**: Discards all service-specific mutual information
  $I(\text{Service}; Y) = 0$.
- **Operational failure**: A catastrophically broken payment gateway test
  ($k_j=50, n_j=50$) is drowned out by 10,000 green unit tests.

### Extreme B: No Pooling (The Overfitting Extremum)
Estimate each $\theta_j$ strictly from its own empirical maximum likelihood
frequency:

$$
\hat{\theta}_j^{\text{MLE}} = \frac{k_j}{n_j}
$$


- **Information cost**: Preserves 100% of empirical noise. The effective degrees
  of freedom equal $J$.
- **Operational failure**: A new test with $k_j=1, n_j=1$ is assigned
  $\hat{\theta}_j = 1.0$ (100% failure rate). A test with $k_j=0, n_j=2$ is
  assigned $\hat{\theta}_j = 0.0$. Both estimates are wildly inaccurate and
  destabilize retry budgets.

```
       [ Complete Pooling ] <========== [ Partial Pooling ] ==========> [ No
Pooling ]
       Over-regularized                 Optimal Compromise
Overfitted
       High Bias                        Min Bayes Risk / Min MSE        High
Variance
       I(Test ID; Theta) = 0            Rate-Distortion Pareto Knee     Max
Empirical Entropy
```



## 2. The Hierarchical Bayes Solution: Adaptive Shrinkage

In a hierarchical model, we do not fix the prior distribution beforehand;
rather, **we learn the prior distribution directly from the fleet of tests**.

### Generative Hierarchical Beta-Binomial Model
In production CI analytics across suites and microservices, tests are grouped by
test suite or component. Rather than setting prior parameters by hand, we
specify a generative hierarchical model across all tests in the suite:

$$
\alpha_{\text{suite}}, \beta_{\text{suite}} \sim \text{Hyperprior}, \qquad \theta_i \sim \text{Beta}(\alpha_{\text{suite}}, \beta_{\text{suite}}), \qquad k_i \sim \text{Binomial}(n_i, \theta_i)
$$

Here, $\alpha_{\text{suite}}$ and $\beta_{\text{suite}}$ govern the overall
flakiness distribution across the suite. In logit space with Gaussian priors,
the equivalent formulation expresses individual test variation as:

$$
\begin{aligned}
\text{logit}(\theta_i) &= \mu + u_i \\
u_i &\sim \mathcal{N}(0, \sigma^2) \\
\mu &\sim \mathcal{N}(\mu_0, \sigma_0^2) \\
\sigma &\sim \text{Half-Normal}(\tau)
\end{aligned}
$$

### The Exact Posterior Shrinkage Formula
Under conjugate Beta-Binomial updating, observing $k_i$ failures in $n_i$
executions of test $i$ yields the posterior distribution $\theta_i \mid D_i \sim
\text{Beta}(\alpha_{\text{suite}} + k_i, \beta_{\text{suite}} + n_i - k_i)$. The
partially pooled posterior mean is:

$$
\tilde{\theta}_i = \frac{\alpha_{\text{suite}} + k_i}{\alpha_{\text{suite}} + \beta_{\text{suite}} + n_i}
$$

We can rewrite this posterior mean as an exact, sample-weighted convex
combination of the **No-Pooling** empirical failure rate $\hat{\theta}_i = k_i /
n_i$ and the **Complete-Pooling** suite baseline mean

$$
\mu_{\text{suite}} = \frac{\alpha_{\text{suite}}}{\alpha_{\text{suite}} + \beta_{\text{suite}}}
$$

with prior concentration $M = \alpha_{\text{suite}} + \beta_{\text{suite}}$ (the
effective prior sample size of the suite):

$$
\tilde{\theta}_i = \underbrace{\left(\frac{n_i}{n_i + M}\right)}_{\text{Data Weight } w_i} \hat{\theta}_i + \underbrace{\left(\frac{M}{n_i + M}\right)}_{\text{Shrinkage } 1 - w_i} \mu_{\text{suite}}
$$

Notice the beautiful and automatic dynamics:
- As $n_i \to \infty$ (a mature, heavily executed test),
$w_i \to 1$ and $1 - w_i \to 0$: empirical data dominates the prior, and
$\tilde{\theta}_i \to \hat{\theta}_i$.
- As $n_i \to 0$ (a brand-new test created on a PR),
$w_i \to 0$ and $1 - w_i \to 1$: empirical noise is suppressed, and the estimate
gracefully shrinks back to the suite baseline $\mu_{\text{suite}}$.

### Worked 3-Test Numerical Table: UI Test Suite
Consider a production UI test suite with baseline mean flakiness
$\mu_{\text{suite}} = 2.0\%$ and concentration $M = 50$, corresponding to
hyperparameters $\alpha_{\text{suite}} = 1.0$ and $\beta_{\text{suite}} = 49.0$.
We evaluate three test endpoints with different execution volumes:

1. **New Test A ($1$ fail in $n=2$ runs):**
   - No-pooling empirical rate: $\hat{\theta}_A = 1/2 = 50.0\%$.
   - Data weight: $w_A = \frac{2}{2 + 50} = \frac{2}{52} \approx 3.8\%$.
   - Prior shrinkage: $1 - w_A = \frac{50}{52} \approx 96.2\%$.
   - Shrunk estimate:

$$
\tilde{\theta}_A = \frac{1 + 1}{50 + 2} = \frac{2}{52} \approx \mathbf{3.85\%}
$$

   - **Operational impact**: No-pooling flags Test A as 50% broken and triggers
a false quarantine, halting PR merges. Hierarchical Bayes recognizes $n=2$ as
tiny sample noise, shrinking it safely to $3.85\%$ and avoiding pipeline
gridlock!

2. **Moderate Test B ($5$ fails in $n=50$ runs):**
   - No-pooling empirical rate: $\hat{\theta}_B = 5/50 = 10.0\%$.
   - Data weight: $w_B = \frac{50}{50 + 50} = 50.0\%$.
   - Prior shrinkage: $1 - w_B = \frac{50}{100} = 50.0\%$.
   - Shrunk estimate:

$$
\tilde{\theta}_B = \frac{1 + 5}{50 + 50} = \frac{6}{100} = \mathbf{6.00\%}
$$

   - **Operational impact**: Equal balance between data and prior.

3. **High-Volume Test C ($50$ fails in $n=500$ runs):**
   - No-pooling empirical rate: $\hat{\theta}_C = 50/500 = 10.0\%$.
   - Data weight: $w_C = \frac{500}{500 + 50} = \frac{500}{550} \approx 90.9\%$.
   - Prior shrinkage: $1 - w_C = \frac{50}{550} \approx 9.1\%$.
   - Shrunk estimate:

$$
\tilde{\theta}_C = \frac{1 + 50}{50 + 500} = \frac{51}{550} \approx \mathbf{9.27\%}
$$

   - **Operational impact**: High sample volume dominates the suite prior.
Test C is genuinely degraded and should be quarantined for deflaking.

| Test Endpoint | Runs $n_i$ | Fails $k_i$ | No-Pooling $\hat{\theta}_i$ | Weight $w_i$ | Shrinkage $1 - w_i$ | Shrunk Mean $\tilde{\theta}_i$ | Operational Action |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Test A (New)** | 2 | 1 | $50.0\%$ | $3.8\%$ | $96.2\%$ | $\mathbf{3.85\%}$ | Route to silent background runs |
| **Test B (Moderate)** | 50 | 5 | $10.0\%$ | $50.0\%$ | $50.0\%$ | $\mathbf{6.00\%}$ | Flag advisory flakiness alert |
| **Test C (High-Vol)** | 500 | 50 | $10.0\%$ | $90.9\%$ | $9.1\%$ | $\mathbf{9.27\%}$ | Quarantine and block merges |

### Connection to the Information Bottleneck
The shrinkage factor $1 - w_i = \frac{M}{n_i + M}$ is the exact filter that
compresses away small-sample binomial sampling noise while preserving genuine
per-test signal. When $n_i$ is small, the empirical outcome contains mostly
sampling variance (nuisance entropy). The hierarchical model compresses this
nuisance information away by regularizing toward the suite prior, minimizing
expected generalization error across the entire fleet.

### The James-Stein Phenomenon and Admissibility
In 1961, Charles Stein proved a counterintuitive theorem that shocked
mathematical statistics: **In three or more dimensions ($J \ge 3$), the
independent sample mean $\hat{\boldsymbol{\theta}}^{\text{MLE}}$ is strictly
inadmissible under squared error loss**.

An estimator that shrinks individual estimates toward a common suite mean
achieves strictly lower total expected mean squared error (MSE) across the
fleet! In conjugate Gaussian-Gaussian settings, the posterior mean is an
analogous weighted balance:

$$
\tilde{\theta}_j = (1 - B_j)\hat{\theta}_j + B_j \bar{\theta}
$$

where the shrinkage factor $B_j \in [0, 1]$ is:

$$
B_j = \frac{\frac{1}{n_j}}{\frac{1}{n_j} + \sigma^2} = \frac{1}{1 + n_j \sigma^2}
$$



## 3. Naftali Tishby's Information Bottleneck Principle

Let us now examine the exact same problem through the lens of Information
Theory.

Let:
- $X$: The raw input identity of the test case ($X \in \{1, 2, \dots, J\}$, high
  cardinality).
- $Y$: The binary outcome of interest ($Y \in \{0, 1\}$, pass vs fail on commit
  merge).
- $T$: A compact, compressed internal representation or latent score assigned to
  the test.

We want our representation $T$ to satisfy two competing objectives:
1. **Compression**: Discard as much irrelevant, idiosyncratic noise about $X$ as
   possible. That is, **minimize** the mutual information $I(X; T)$.
2. **Relevance / Retention**: Preserve as much information about the target
   outcome
   $Y$ as possible. That is, **maximize** the mutual information $I(T; Y)$.

```
   Raw Test ID (X) --------[ Bottleneck p(t|x) ]--------> Latent Embedding (T)
      |                                                        |
      |   (Nuisance Variations, Small-Sample Noise)            | (Preserved
Signal)
      v                                                        v
   Discarded: I(X; T) minimized                         Retained: I(T; Y)
maximized
                                                               |
                                                               v
                                                       Predicted Failure (Y)
```

Naftali Tishby formalized this tradeoff as a variational Lagrangian optimization
problem:

$$
\min_{p(t|x)} \mathcal{L}_{\text{IB}}[p(t|x)] = I(X; T) - \beta I(T; Y)
$$


where $\beta > 0$ is the Lagrange multiplier governing the exchange rate between
bits of compression and bits of predictive power.

### The Information Curve (The Rate-Distortion Frontier)
When we sweep $\beta$ from $0$ to $\infty$, we trace the **Information
Bottleneck Curve** in the plane $(I(X; T), I(T; Y))$:

- **$\beta \to 0$**: We care only about compression. $T$ is independent of $X$.
  $I(X; T) = 0$, but $I(T; Y) = 0$. (Complete Pooling).
- **$\beta \to \infty$**: We care only about prediction on training samples. $T
  =
  X$. $I(X; T) = H(X)$ (maximum bits), $I(T; Y) = I(X; Y)$. (No Pooling /
  Overfitting).
- **$0 < \beta < \infty$**: The concave Pareto frontier where every bit of
  complexity $I(X; T)$ bought yields the maximum possible predictive return
  $I(T; Y)$.

```
  I(T; Y) ^
          |                 Optimal Pareto Curve (Concave)
   I(X;Y) +------------------------* (No Pooling: High Noise, Max Bits)
          |                   .---|
          |              .---'
          |          .--'  <-- "Knee of the Curve" (Optimal Hierarchical
Shrinkage)
          |       .-'
          |     .'
        0 +----+---------------------------------------------> I(X; T)
          (Complete Pooling: 0 bits)                        H(X)
```



## 4. The Deep Equivalence: Partial Pooling IS an Information Bottleneck

Why do Bayesian hierarchical models and Information Bottleneck theory arrive at
the same operational conclusion?

Consider a hierarchical prior with hyperparameter variance $\sigma^2$:
- When $\sigma^2 \to 0$, all group effects collapse to $0$: $u_j \equiv 0$. The
  latent state $T$ is a constant. Mutual information $I(X; T) = 0$.
- When $\sigma^2 \to \infty$, the prior places zero constraint on $u_j$. The
  latent state stores all $J$ distinct parameters. $I(X; T) = \log_2(J)$.

In fact, the Information Bottleneck objective:

$$
\mathcal{L}_{\text{IB}} = I(X; T) - \beta I(T; Y)
$$

is mathematically isomorphic to the Evidence Lower Bound (ELBO) in a
$\beta$-Variational Autoencoder ($\beta$-VAE):

$$
\text{ELBO}_{\beta} = \mathbb{E}_{q(z|x)}[\ln p(y|z)] - \beta
D_{\text{KL}}(q(z|x) \parallel p(z))
$$


The regularizing KL divergence $D_{\text{KL}}(q(z|x) \parallel p(z))$ measures
the number of bits required to encode the individual test's identity beyond the
fleet prior $p(z)$.

**Hierarchical partial pooling is Rate-Distortion theory applied to statistical
inference.**



## 5. What Do We Actually Do With This Number?
### From Wasted Bits to Operational Action

Every theoretical quantity in this chapter converts directly into a CI/CD
operational policy.

### Action 1: Dynamic Quarantine Sizing via Shrinkage Variance
- **The Problem**: A developer introduces a new test. Over its first 5 runs, it
  fails twice ($k=2, n=5$). The naive failure rate is $40\%$. Quarantining tests
  blocks PR merges; not quarantining lets bad tests into `main`.
- **The Operational Rule**: Compute the posterior standard deviation under the
  hierarchical model $\sigma_{\text{post}, j}$.
  - If the shrinkage estimate $\tilde{\theta}_j > \theta_{\text{threshold}}$
    **AND** the posterior entropy $h(\theta_j) < h_{\text{certainty}}$,
    **quarantine the test immediately**.
  - If $\tilde{\theta}_j > \theta_{\text{threshold}}$ but $h(\theta_j)$ is high
    (insufficient sample size, heavy shrinkage toward the fleet average
    $\bar{\theta} = 2\%$), **do NOT quarantine**. Instead, route the test to a
    background shadow-runner that executes 30 silent runs off the critical path.

### Action 2: Tuning $\beta$ as Compute-Budget Allocation
- **The Problem**: Running 50,000 tests on every commit costs \$1.2M annually in
  AWS compute.
- **The Operational Rule**: Compute the fleet Information Bottleneck curve $I(T;
  Y)$ vs $I(X; T)$.
  - Find the "knee" $\beta^* = \frac{d I(X; T)}{d I(T; Y)}$.
  - Group tests into $K$ compressed clusters (latent risk tiers $T$).
  - Fast-path tests with $T=0$ (dormant unit tests with near-zero historical
    failure entropy) are skipped or run on a 1-in-10 sampling schedule.
  - High-information tests with $T=1$ (brittle end-to-end integration tests)
    receive dedicated CI slots.
  - **Result**: 60% reduction in total CI core-hours with $<0.1\%$ increase in
    uncaught regression escapes.

### Action 3: Fleet Prior Re-Anchoring During Monorepo Upgrades
- **The Problem**: A major runtime upgrade (e.g., Python 3.11 to 3.12, or Node
  18
  to 20) shifts the entire fleet baseline.
- **The Operational Rule**: Monitor the fleet hyperparameter posterior $p(\mu,
  \sigma^2 \mid \mathcal{D})$.
  - If the fleet variance $\sigma^2$ spikes by $>2\times$, it indicates that
    failure modes are fracturing across teams (heterogeneity increased).
  - Automatically tighten the shrinkage prior by increasing the pseudo-run
    hyperparameter weight to prevent false alarms until the fleet restabilizes.



## 6. Summary and Transition

| Regime | Parameter Count | Mutual Information $I(X; T)$ | Expected Generalization Error | Operational CI Effect |
| :--- | :--- | :--- | :--- | :--- |
| **Complete Pooling** | 1 ($\theta_{\text{fleet}}$) | $0$ bits | High (Underfitting / High Bias) | Rogue broken tests escape into production |
| **Partial Pooling (Hierarchical)** | $J$ shrunk parameters | $I^*(X; T) \in (0, \log J)$ | **Minimal (Optimal Bayes Risk)** | Noise suppressed; real regressions flagged |
| **No Pooling (MLE)** | $J$ independent parameters | $\log_2(J)$ bits | High (Overfitting / High Variance) | False alarm storms on newly added tests |

In the next chapter (**Chapter 08: Model Comparison as Code Efficiency**), we
will confront a vital question: *How do we know if our hierarchical model is
actually better than a simpler flat model?* We will prove that out-of-sample
Bayesian model comparison via WAIC and PSIS-LOO is mathematically equivalent to
optimizing cross-entropy code length.

